# 📡 Aula 22 — Automação e Monitoramento do Projeto

**Curso:** Técnico em Inteligência Artificial | **UC:** Introdução a Arquitetura de Computadores

---

## 🎯 Situação de Aprendizagem

O modelo está treinando e os primeiros resultados são promissores. Mas os treinamentos agora duram **horas ou dias**. Ninguém ficará olhando o terminal 24h por dia. O time precisa de um sistema que: colete métricas da GPU continuamente, dispare **alertas automáticos** se a temperatura ou a VRAM passarem dos limites, gere **dashboards** e integre tudo ao repositório do projeto — sem intervenção manual.

Este notebook roda **sem GPU** (usa um simulador de métricas para exercitar o pipeline).

In [ ]:
# 1. Verificar GPU e importar dependências básicas
import shutil, os, random, time

TEM_NVIDIA_SMI = shutil.which("nvidia-smi") is not None
print(f"nvidia-smi disponível: {TEM_NVIDIA_SMI}")
print("Sem GPU: usaremos um gerador de métricas simuladas com o mesmo formato do CSV real.")

## 1. CSV estruturado de métricas

O mesmo formato de log da Aula 14, agora com o campo `treino_ativo` para correlacionar com o treino.

In [ ]:
# 2. Simular um log CSV de monitoramento (mesmo cabeçalho do script Bash)
import csv

CABECALHO = "timestamp,gpu_idx,temp_c,power_w,util_pct,mem_used_mb,mem_total_mb,mem_pct,treino_ativo".split(",")
LOG_FILE = "gpu_monitor_demo.csv"

def gerar_amostra(i, treino_ativo=1):
    """Gera uma linha simulada: aquece ao longo do tempo para disparar alertas."""
    temp = 60 + int(i * 2.5) + random.randint(-2, 2)
    util = 70 + random.randint(0, 29)
    mem_used = 8000 + i * 200
    mem_total = 15360
    mem_pct = int(mem_used * 100 / mem_total)
    pw = 180 + random.randint(0, 80)
    ts = time.strftime("%Y-%m-%dT%H:%M:%S")
    return [ts, 0, temp, pw, util, mem_used, mem_total, mem_pct, treino_ativo]

with open(LOG_FILE, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(CABECALHO)
    for i in range(12):
        w.writerow(gerar_amostra(i))

print(f"Log gerado: {LOG_FILE}")
with open(LOG_FILE, encoding="utf-8") as f:
    print(f.read())

## 2. Alertas por limiar

Reproduzimos a lógica do `alertas.sh`: WARN quando passa de 82 °C / 85% VRAM e CRITICAL a 90 °C / 95%.

In [ ]:
# 3. Analisar o CSV e gerar alertas graduais
TEMP_WARN, TEMP_CRIT = 82, 90
VRAM_WARN, VRAM_CRIT = 85, 95

def analisar_alertas(csv_file):
    alertas = []
    with open(csv_file, encoding="utf-8") as f:
        for linha in csv.DictReader(f):
            temp = int(linha["temp_c"])
            mem = int(linha["mem_pct"])
            if temp >= TEMP_CRIT:
                alertas.append((linha["timestamp"], "CRITICAL", f"temperatura {temp}C"))
            elif temp >= TEMP_WARN:
                alertas.append((linha["timestamp"], "WARN", f"temperatura {temp}C"))
            if mem >= VRAM_CRIT:
                alertas.append((linha["timestamp"], "CRITICAL", f"VRAM {mem}%"))
            elif mem >= VRAM_WARN:
                alertas.append((linha["timestamp"], "WARN", f"VRAM {mem}%"))
    return alertas

alertas = analisar_alertas(LOG_FILE)
for ts, nivel, msg in alertas:
    print(f"[{ts}] {nivel:<8} {msg}")
print(f"\nTotal de alertas: {len(alertas)}")

## 3. Alertas multi-canal (simulação)

Em produção, a função abaixo envia para Slack/Telegram. Aqui, apenas montamos o **payload** que seria enviado.

In [ ]:
# 4. Montar o payload de um alerta (sem enviar de verdade)
import json

def montar_payload_slack(nivel, mensagem, hostname="servidor-gpu"):
    emoji = {"INFO": "i", "WARN": "!" , "CRITICAL": "!!"}.get(nivel, "?")
    return {"text": f"{emoji} *{nivel}* - {hostname}\n{mensagem}"}

payload = montar_payload_slack("CRITICAL", "GPU0: temperatura 92C - RISCO DE THROTTLING")
print("Payload Slack:")
print(json.dumps(payload, ensure_ascii=False, indent=2))
print("\nEm produção: requests.post(SLACK_WEBHOOK_URL, json=payload)")

## 4. Dashboard com pandas + matplotlib

As 4 métricas principais em um relatório visual.

In [ ]:
# 5. Gerar o dashboard a partir do CSV
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(LOG_FILE, parse_dates=["timestamp"])
df["duracao_min"] = (df["timestamp"] - df["timestamp"].iloc[0]).dt.total_seconds() / 60

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle("Dashboard de Monitoramento GPU (demo)", fontsize=14, fontweight="bold")

axes[0, 0].plot(df["duracao_min"], df["temp_c"], color="#EF4444", label="Temp C")
axes[0, 0].axhline(TEMP_WARN, color="orange", ls="--", label="Warn 82C")
axes[0, 0].axhline(TEMP_CRIT, color="red", ls="--", label="Crit 90C")
axes[0, 0].set_title("Temperatura (C)"); axes[0, 0].legend()

axes[0, 1].plot(df["duracao_min"], df["util_pct"], color="#10B981")
axes[0, 1].set_ylim(0, 100); axes[0, 1].set_title("Utilizacao GPU (%)")

axes[1, 0].fill_between(df["duracao_min"], df["mem_used_mb"] / 1024, alpha=0.3, color="#7E22CE")
axes[1, 0].plot(df["duracao_min"], df["mem_used_mb"] / 1024, color="#7E22CE")
axes[1, 0].set_title("VRAM Usada (GB)")

axes[1, 1].plot(df["duracao_min"], df["power_w"], color="#F97316")
axes[1, 1].set_title("Potencia (W)")

plt.tight_layout()
plt.savefig("dashboard_demo.png", dpi=120)
plt.show()

## 5. Integração com o train.py via subprocess

O monitor roda em processo separado e encerra junto com o treino (bloco `finally`).

In [ ]:
# 6. Padrão de integração do monitor (esqueleto didático)
from dataclasses import dataclass, field

@dataclass
class Monitor:
    """Representa o monitor externo iniciado junto do treino."""
    nome_projeto: str
    intervalo: int = 10
    ativo: bool = field(default=False)

    def iniciar(self):
        self.ativo = True
        print(f"[Monitor] iniciado para '{self.nome_projeto}' (intervalo {self.intervalo}s)")

    def encerrar(self):
        self.ativo = False
        print(f"[Monitor] encerrado para '{self.nome_projeto}'")

def treino_simulado(monitor, epocas=3):
    try:
        for ep in range(1, epocas + 1):
            print(f"  [Treino] epoca {ep}/{epocas}")
            time.sleep(0.2)
    finally:
        monitor.encerrar()   # garante o encerramento mesmo com erro

mon = Monitor("DeteccaoDoencas")
mon.iniciar()
treino_simulado(mon)
print(f"Monitor ainda ativo? {mon.ativo}")

---
## 📝 6. Exercícios Práticos (5)

Resolva os 5 exercícios a seguir completando as células de código correspondentes.

### Exercício 1: Gerar um log mais longo

Gere 30 amostras em `gpu_monitor_ex1.csv` e conte quantas dispararam **CRITICAL** de temperatura.

In [ ]:
# Escreva a sua solução para o Exercício 1 abaixo:
# reuse gerar_amostra(i) e csv.writer, depois analisar_alertas(...)

### Exercício 2: Ajustar os thresholds

Mude `TEMP_WARN` para 75 e `TEMP_CRIT` para 85. Quantos alertas são gerados agora? Compare com o original.

In [ ]:
# Escreva a sua solução para o Exercício 2 abaixo:
# TEMP_WARN, TEMP_CRIT = 75, 85 ; print(len(analisar_alertas(LOG_FILE)))

### Exercício 3: Payload de alerta de VRAM

Use `montar_payload_slack` para montar um alerta **WARN** de VRAM em 88% e imprima o JSON.

In [ ]:
# Escreva a sua solução para o Exercício 3 abaixo:
# print(json.dumps(montar_payload_slack('WARN', 'GPU0: VRAM 88%'), indent=2))

### Exercício 4: Estatísticas do log

Com pandas, imprima a temperatura **máxima**, **média** e a VRAM **de pico** (GB) do log de demonstração.

In [ ]:
# Escreva a sua solução para o Exercício 4 abaixo:
# print(df['temp_c'].max(), df['temp_c'].mean(), df['mem_used_mb'].max()/1024)

### Exercício 5: Monitor que encerra em erro

Simule um treino que lança uma exceção na 2ª época e confirme, com `mon.ativo`, que o monitor foi encerrado mesmo assim.

In [ ]:
# Escreva a sua solução para o Exercício 5 abaixo:
# def treino_com_erro(monitor): ... raise RuntimeError('falha')

---
## 📌 7. Síntese e Conclusão

- **Monitor integrado:** `subprocess.Popen` + `finally` garante que o monitor vive e morre com o treinamento.
- **CSV estruturado:** timestamp como índice permite unir com as métricas do W&B.
- **Alertas multi-canal:** Slack é o mais rápido para times; Telegram para alertas pessoais.
- **Thresholds graduais:** WARN a 82 °C/85% VRAM e CRITICAL a 90 °C/95%.
- **Dashboard automático:** gere o PNG ao fim de cada sessão e commite junto dos resultados.
- **systemd em produção:** monitoramento contínuo independente do treinamento.

**Tarefa para a Aula 23:** treinar com o monitor integrado, gerar o dashboard final e preparar o pitch de 5 minutos.